In [1]:
!pip install -U albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.0/224.0 kB 8.8 MB/s eta 0:00:00
  Attempting uninstall: albumentations
    Found existing installation: albumentations 1.4.17
    Uninstalling albumentations-1.4.17:
      Successfully uninstalled albumentations-1.4.17


In [2]:
import os

import cv2

import numpy as np

import xml.etree.ElementTree as ET

import albumentations as A

C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\albumentations\__init__.py:13: UserWarning: A new version of Albumentations is available: 1.4.20 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [3]:
#Path
save_dir = 'Data_Set_Larch_Casebearer/Croppedimages'

In [4]:
def parse_xml(xml_file):

    tree = ET.parse(xml_file)

    root = tree.getroot()



    bboxes = []

    labels = []



   

    for obj in root.findall('object'):

        # Check if the 'damage' tag exists

        damage_tag = obj.find('damage')

        if damage_tag is not None:

            label = damage_tag.text  # Healthy, Light Damage, High Damage, Others

        else:

            label = 'unknown'  # Handle the case where 'damage' tag is missing



        # Parse bounding box coordinates

        bndbox = obj.find('bndbox')

        if bndbox is not None:

            xmin = int(bndbox.find('xmin').text)

            ymin = int(bndbox.find('ymin').text)

            xmax = int(bndbox.find('xmax').text)

            ymax = int(bndbox.find('ymax').text)

            bboxes.append([xmin, ymin, xmax, ymax])

            labels.append(label)



    return bboxes, labels

In [4]:
import os

import cv2



# Function to crop the images based on bounding boxes and save them

def crop_and_save_images(image_dir, annotation_dir, save_dir):

    # Iterate over all images and corresponding annotations

    for img_file in os.listdir(image_dir):

        if img_file.endswith('.JPG') or img_file.endswith('.png'):

            img_path = os.path.join(image_dir, img_file)

            

            # Corresponding XML annotation file

            xml_file = os.path.join(annotation_dir, os.path.splitext(img_file)[0] + '.xml')

            

            # Check if the XML file exists

            if os.path.exists(xml_file):

                # Read image

                img = cv2.imread(img_path)

                

                # Check if the image is loaded correctly

                if img is None:

                    print(f"Error: Unable to load image {img_path}")

                    continue  # Skip this image and continue with the next one



                # Parse the XML file to get bounding boxes and labels

                bboxes, labels = parse_xml(xml_file)



                height, width, _ = img.shape

                

                # Crop and save the trees

                for i, bbox in enumerate(bboxes):

                    xmin, ymin, xmax, ymax = bbox



                    # Validate bounding boxes (make sure they are within the image boundaries)

                    if xmin < 0 or ymin < 0 or xmax > width or ymax > height:

                        print(f"Warning: Bounding box {bbox} is out of bounds for image {img_file}")

                        continue  # Skip this bounding box



                    # Crop the region defined by the bounding box

                    cropped_img = img[ymin:ymax, xmin:xmax]



                    # Ensure the cropped image is not empty

                    if cropped_img.size > 0:

                        label = labels[i]



                        # Ensure save directory exists

                        if not os.path.exists(save_dir):

                            os.makedirs(save_dir)



                        # Save cropped image based on label

                        cropped_save_path = os.path.join(save_dir, f'{label}_{os.path.splitext(img_file)[0]}_{i}.jpg')

                        cv2.imwrite(cropped_save_path, cropped_img)

                    else:

                        continue

            else:

                continue


In [5]:
crop_   and_save_images(image_dir, annotation_dir , save_dir)

print("Images are cropped!!")

Images are cropped!!


In [4]:
import pandas as pd

In [5]:
image_files=os.listdir(save_dir)

def extract_label(filename):

    if filename.split('_')[0]=='H':

        return "Healthy"

    elif filename.split('_')[0]=='LD':

        return "Light Damage"

    elif filename.split('_')[0]=='HD':

        return "Heavily Damaged"

data={'file_path':[os.path.join(save_dir,file)for file in image_files],

     'label':[extract_label(file) for file in image_files]}

df=pd.DataFrame(data)

In [6]:
print(df)

                                               file_path            label
0      Data_Set_Larch_Casebearer/Croppedimages\.ipynb...             None
1      Data_Set_Larch_Casebearer/Croppedimages\HD_B01...  Heavily Damaged
2      Data_Set_Larch_Casebearer/Croppedimages\HD_B01...  Heavily Damaged
3      Data_Set_Larch_Casebearer/Croppedimages\HD_B01...  Heavily Damaged
4      Data_Set_Larch_Casebearer/Croppedimages\HD_B01...  Heavily Damaged
...                                                  ...              ...
45018  Data_Set_Larch_Casebearer/Croppedimages\LD_B05...     Light Damage
45019  Data_Set_Larch_Casebearer/Croppedimages\LD_B05...     Light Damage
45020  Data_Set_Larch_Casebearer/Croppedimages\unknow...             None
45021  Data_Set_Larch_Casebearer/Croppedimages\unknow...             None
45022  Data_Set_Larch_Casebearer/Croppedimages\unknow...             None

[45023 rows x 2 columns]


In [7]:
# Filter the dataframe to keep only rows with specific labels
valid_labels = ['Light Damage', 'Healthy', 'Heavily Damaged']
df = df[df['label'].isin(valid_labels)]  # Exclude None 

In [8]:
print("Unique labels in the dataset:", df['label'].unique())

Unique labels in the dataset: ['Heavily Damaged' 'Healthy' 'Light Damage']


In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [10]:
datagen=ImageDataGenerator(

    rescale=1.0/255.0,

    validation_split=0.2

)

train_df=df.sample(frac=0.8,random_state=42)

val_df=df.drop(train_df.index)




In [11]:
# Training generator

train_generator = datagen.flow_from_dataframe(

    train_df,

    x_col='file_path',

    y_col='label',

    target_size=(224, 224),  # Adjust based on model input

    batch_size=16,

    class_mode='categorical',

    subset='training'

)



# Validation generator

validation_generator = datagen.flow_from_dataframe(

    val_df,

    x_col='file_path',

    y_col='label',

    target_size=(224, 224),

    batch_size=16,

    class_mode='categorical',

    subset='validation'

)

Found 28812 validated image filenames belonging to 3 classes.
Found 1800 validated image filenames belonging to 3 classes.


In [12]:
import tensorflow as tf
from tensorflow.keras import layers, models
import timm 


In [63]:
import tensorflow as tf

import tensorflow_hub as hub




@tf.keras.saving.register_keras_serializable()
class SwinClassifier(tf.keras.Model):

    def __init__(self, num_classes, swin_url):

        super(SwinClassifier, self).__init__()

        self.swin = hub.KerasLayer(swin_url, trainable=False, name="swin_transformer")

        self.dense1 = tf.keras.layers.Dense(512, activation='relu', name="dense_512")

        self.dropout = tf.keras.layers.Dropout(0.5, name="dropout_0.5")

        self.output_layer = tf.keras.layers.Dense(num_classes, activation='softmax', name="output")



    def call(self, inputs):

        x = self.swin(inputs)
        
        x = self.dense1(x)

        x = self.dropout(x)

        return self.output_layer(x)



    def build(self, input_shape):

        super().build(input_shape)

        self.build_graph = tf.function(

            self.call,

            input_signature=[tf.TensorSpec(shape=input_shape, dtype=tf.float32)]

        )
swin_url = "https://tfhub.dev/sayakpaul/swin_tiny_patch4_window7_224_fe/1"
model = SwinClassifier(num_classes=3, swin_url=swin_url)

In [67]:
# Compile the model

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy',tf.keras.metrics.Precision(name='precision'),tf.keras.metrics.Recall(name='recall')])
sample_input = tf.random.normal((1, 224, 224, 3))
_ = model(sample_input)
model.summary()

Inputs shape: (1, 224, 224, 3)
After SWIN shape: (1, 768)


Model: "swin_classifier_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_512 (Dense)                    │ (1, 512)                    │         393,728 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_0.5 (Dropout)                │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ output (Dense)                       │ (1, 3)                      │           1,539 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 395,267 (1.51 MB)

 Trainable params: 395,267 (1.51 MB)

 Non-trainable params: 0 (0.00 B)

In [68]:
import tensorflow as tf
policy = tf.keras.mixed_precision.set_global_policy('mixed_float16')

In [71]:
model.fit(

    train_generator,

    steps_per_epoch=train_generator.samples //64,

    validation_data=validation_generator,

    validation_steps=validation_generator.samples //64,

    epochs=10

)



Epoch 1/10
450/450 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7455 - loss: 0.6232 - precision: 0.7502 - recall: 0.7396Inputs shape: (None, 224, 224, 3)
After SWIN shape: (None, 768)
450/450 ━━━━━━━━━━━━━━━━━━━━ 887s 2s/step - accuracy: 0.7455 - loss: 0.6232 - precision: 0.7502 - recall: 0.7397 - val_accuracy: 0.4129 - val_loss: 0.9846 - val_precision: 0.4079 - val_recall: 0.3906
Epoch 2/10
450/450 ━━━━━━━━━━━━━━━━━━━━ 888s 2s/step - accuracy: 0.7841 - loss: 0.5163 - precision: 0.7890 - recall: 0.7780 - val_accuracy: 0.3438 - val_loss: 1.1848 - val_precision: 0.3425 - val_recall: 0.3348
Epoch 3/10
450/450 ━━━━━━━━━━━━━━━━━━━━ 888s 2s/step - accuracy: 0.7843 - loss: 0.4811 - precision: 0.7892 - recall: 0.7792 - val_accuracy: 0.2879 - val_loss: 1.1873 - val_precision: 0.2818 - val_recall: 0.2768
Epoch 4/10
450/450 ━━━━━━━━━━━━━━━━━━━━ 885s 2s/step - accuracy: 0.7903 - loss: 0.4624 - precision: 0.7911 - recall: 0.7844 - val_accuracy: 0.6451 - val_loss: 0.6853 - val_precision: 0.6471 - v

C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(value)


450/450 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9375 - loss: 0.2532 - precision: 0.9375 - recall: 0.9375 - val_accuracy: 0.6250 - val_loss: 0.6435 - val_precision: 0.6250 - val_recall: 0.6250
Epoch 6/10
450/450 ━━━━━━━━━━━━━━━━━━━━ 899s 2s/step - accuracy: 0.8122 - loss: 0.4378 - precision: 0.8132 - recall: 0.8093 - val_accuracy: 0.5670 - val_loss: 0.8369 - val_precision: 0.5636 - val_recall: 0.5536
Epoch 7/10
450/450 ━━━━━━━━━━━━━━━━━━━━ 886s 2s/step - accuracy: 0.8005 - loss: 0.4402 - precision: 0.8031 - recall: 0.7982 - val_accuracy: 0.4844 - val_loss: 0.8882 - val_precision: 0.4820 - val_recall: 0.4777
Epoch 8/10
450/450 ━━━━━━━━━━━━━━━━━━━━ 895s 2s/step - accuracy: 0.7991 - loss: 0.4450 - precision: 0.8005 - recall: 0.7948 - val_accuracy: 0.4844 - val_loss: 0.8828 - val_precision: 0.4831 - val_recall: 0.4799
Epoch 9/10
450/450 ━━━━━━━━━━━━━━━━━━━━ 888s 2s/step - accuracy: 0.8113 - loss: 0.4254 - precision: 0.8119 - recall: 0.8079 - val_accuracy: 0.4799 - val_loss: 0.9214 - 

In [20]:
model.save('foresttree_health_classification_model.keras')

In [21]:
test_df=df.sample(frac=0.1,random_state=42)

In [25]:
test_datagen=ImageDataGenerator(rescale=1./255.)
test_generator = test_datagen.flow_from_dataframe(

    test_df,

    x_col='file_path',

    y_col='label',

    target_size=(224, 224),  # Adjust based on model input

    batch_size=8,

    class_mode='categorical'

)
test_loss, test_acc = model.evaluate(test_generator, steps=test_generator.samples // 8)
print(f'Test Accuracy: {test_acc:.4f}')

Found 4502 validated image filenames belonging to 3 classes.


C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


562/562 ━━━━━━━━━━━━━━━━━━━━ 459s 817ms/step - accuracy: 0.8225 - loss: 0.4034
Test Accuracy: 0.8247


In [26]:
import random
import cv2
import numpy as np
random_files = random.sample(list(df['file_path']), 3)
input_images = []
for file in random_files:
    # Load image
    img = cv2.imread(file)
    img = cv2.resize(img, (224, 224))
    img = img / 255.0
    input_images.append(img)


input_images = np.array(input_images)
predictions = model.predict(input_images)


class_labels = train_generator.class_indices
label_map = {v: k for k, v in class_labels.items()}  # reverse key-value pairs

for idx, pred in enumerate(predictions):
    predicted_class = label_map[np.argmax(pred)]
    print(f"Image {random_files[idx]} is classified as: {predicted_class}")


Inputs shape: (3, 224, 224, 3)
After SWIN shape: (3, 768)
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Image Data_Set_Larch_Casebearer/Croppedimages\LD_B04_0101_115.jpg is classified as: Light Damage
Image Data_Set_Larch_Casebearer/Croppedimages\LD_B04_0055_56.jpg is classified as: Light Damage
Image Data_Set_Larch_Casebearer/Croppedimages\LD_B04_0125_27.jpg is classified as: Light Damage
